In [33]:
import pandas as pd
import os

# === CONFIGURATION ===
INPUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\Instru_Analysis_V2.0\3.1_Config_Files_List_ShallowC.csv"
OUTPUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\Instru_Analysis_V2.0\3.1_Repo_Summary.csv"
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

# === LOAD DATA ===
df = pd.read_csv(INPUT_CSV)

# === CLEAN CI PLATFORM ===
df['ci_platform_clean'] = (
    df['ci_platform']
    .fillna("Unknown")
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({'': 'unknown'})
)

# === CLEAN DEVICE SETUP ===
df['device_setup_clean'] = df['device_setup'].fillna("None").astype(str).str.strip()
df['test_definition_clean'] = df['test_definition'].fillna("None").astype(str).str.strip()
df['test_trigger_clean'] = df['test_trigger'].fillna("None").astype(str).str.strip()

# === CLEAN INSTRU TEST STATUS ===
df['instru_test_status_clean'] = df['instru_test_status'].fillna("None").astype(str).str.strip()

# === HELPERS ===
def aggregate_platforms(series):
    vals = sorted(set(v for v in series if v != 'other'))
    return ", ".join(vals) if vals else "None"

def aggregate_device_setup(series):
    vals = [v for v in series if v != "None"]
    return ", ".join(sorted(set(vals))) if vals else "None"

def aggregate_test_trigger(series):
    vals = [v for v in series if v != "None"]
    return ", ".join(sorted(set(vals))) if vals else "None"

def aggregate_test_definition(series):
    vals = [v for v in series if v != "None"]
    return ", ".join(sorted(set(vals))) if vals else "None"


def aggregate_instru_total(series):
    vals = [v for v in series if v != "None"]
    return ", ".join(sorted(set(vals))) if vals else "None"

def priority_instru_status(series):
    if "Complete" in series.values:
        return "Complete"
    elif "Manual" in series.values:
        return "Manual"
    elif "Defective" in series.values:
        return "Defective"
    else:
        return "None"

# === GROUP AND AGGREGATE BY REPO ===
repo_summary = df.groupby('full_name').agg({
    'ci_platform_clean': aggregate_platforms,        # all unique CI platforms except "other"
    'test_definition_clean': aggregate_test_definition,     # aggregated test definitions
    'test_trigger_clean': aggregate_test_trigger,           # aggregated test triggers
    'device_setup_clean': aggregate_device_setup,           # aggregated device setup

    'has_test_definition': lambda x: 'True' if any(v == 'Yes' for v in x) else 'False',
    'has_device_setup': lambda x: 'True' if any(v == 'Yes' for v in x) else 'False',
    'has_trigger': lambda x: 'True' if any(v == 'Yes' for v in x) else 'False',
    'instru_test_status_clean': aggregate_instru_total,   # aggregated instru statuses
}).reset_index()

# === RENAME COLUMNS ===
repo_summary = repo_summary.rename(columns={
    'ci_platform_clean': 'repo_ci_platforms',
    'test_definition_clean': 'repo_test_definition',
    'device_setup_clean': 'repo_device_setup',
    'test_trigger_clean': 'repo_test_trigger',
    'has_test_definition': 'repo_has_test_definition',
    'has_device_setup': 'repo_has_device_setup',
    'has_trigger': 'repo_has_trigger',
    'instru_test_status_clean': 'repo_instru_test_total'
})

# === ADD PRIORITY STATUS COLUMN ===
repo_summary['repo_instru_test_status'] = df.groupby('full_name')['instru_test_status_clean'].apply(priority_instru_status).values

# === PIVOT CI PLATFORM COUNTS (exclude "other") ===
ci_df = df[df['ci_platform_clean'] != 'other']
ci_counts = pd.crosstab(ci_df['full_name'], ci_df['ci_platform_clean']).reset_index()
ci_counts = ci_counts.rename(columns={col: f"ci_p_{col}" for col in ci_counts.columns if col != 'full_name'})
ci_counts['ci_platform_total'] = ci_counts.drop(columns=['full_name']).sum(axis=1)

# === MERGE EVERYTHING ===
repo_summary = repo_summary.merge(ci_counts, on='full_name', how='left')

# === SAVE OUTPUT ===
repo_summary.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Aggregated repository summary with CI platforms, device setup, and instru test statuses saved to:\n{OUTPUT_CSV}")


✅ Aggregated repository summary with CI platforms, device setup, and instru test statuses saved to:
C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\Instru_Analysis_V2.0\3.1_Repo_Summary.csv


In [5]:
# Create the initial list of Cloned Repositories

import pandas as pd
import os

# === File path ===
file_path = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\Clone_Status.csv"
output_path = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\Instru_Analysis_V2.0\3.4_Total_Repo.csv"

# Read CSV
df = pd.read_csv(file_path)

# Filter repos where clone_status == 'yes'
filtered_df = df[df['clone_status'].str.lower() == 'yes'].copy()

# Create 'full_name' column from the html_url (extract owner/repo)
filtered_df['full_name'] = filtered_df['html_url'].str.replace(
    r"https://github.com/", "", regex=True
)

# Save output (optional)

filtered_df.to_csv(output_path, index=False)

print(f"The filtered list named 3.4_Total_Repo is saved at: {output_path}")



The filtered list named 3.4_Total_Repo is saved at: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\Instru_Analysis_V2.0\3.4_Total_Repo.csv
